# CropIQ — Model Analysis

Loads the trained `models/best_model.pkl` and the held-out test split, reports
headline metrics, breakdowns by state and year, and renders the SHAP-driven figures
in `reports/figures/`.

This notebook is the source of `reports/results.md` content. Re-run after
`make train` to refresh the figures and the markdown.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from src.concepts.feature_matrix import load as load_features
from src.concepts.yield_predictor import load_model, split_by_year
from src.concepts.evaluation import Evaluation
from src.concepts.explanation import make_tree_explainer, plot_waterfall, explain_global
from src.config import settings

%matplotlib inline


In [ ]:
features = load_features(PROJECT_ROOT / 'data/processed/features.parquet')
payload = load_model(PROJECT_ROOT / 'models/best_model.pkl')
model = payload['model']
feature_schema = payload['metadata']['feature_schema']
print('model:', payload['kind'])
print('feature_count:', len(feature_schema))
print('residual_std:', payload['metadata']['residual_std'])


In [ ]:
splits = split_by_year(
    features,
    train_year_max=settings.train_year_max,
    val_years=settings.val_years,
    test_years=settings.test_years,
    feature_columns=feature_schema,
)
X_train, y_train, X_val, y_val, X_test, y_test = splits
preds = model.predict(X_test)
test_rows = features.loc[features['year'].isin(settings.test_years), ['county_fips', 'year', 'state']].reset_index(drop=True)
evl = Evaluation.from_arrays(y_test.to_numpy(), preds, test_rows)
print(evl.compute())


In [ ]:
by_state = evl.breakdown_by('state')
by_year = evl.breakdown_by('year')
by_state


In [ ]:
FIG = PROJECT_ROOT / 'reports/figures'
evl.plot_predicted_vs_actual(FIG / 'pred_vs_actual.png')
evl.plot_residuals(FIG / 'residuals.png')
print('Saved pred_vs_actual.png + residuals.png')


In [ ]:
# SHAP global summary on a 200-row sample (gotcha #7)
explain_global(model, X_test, sample_size=200, path=FIG / 'shap_summary.png')
print('Saved shap_summary.png')


In [ ]:
# Five SHAP waterfalls — choose a spread of yield outcomes
explainer = make_tree_explainer(model)
n = len(X_test)
picks = [int(i) for i in np.linspace(0, n - 1, num=5)]
for k, idx in enumerate(picks, start=1):
    plot_waterfall(explainer, X_test.iloc[[idx]], FIG / f'shap_waterfall_{k}.png', title=f'SHAP waterfall — example #{k}')
print('Saved 5 shap_waterfall_*.png')
